In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## Using the model to make predictions

In [1]:
import pandas as pd

# condition = "control"
# condition = "treatment"
# input_data = pd.read_csv(f"./all_{condition}_seekerhelper_pairs.csv")
input_data = pd.read_csv("N94_all_seekerhelper_pairs.csv")
# input_data = pd.read_csv("N94_all_seekerhelperalternative.csv")
print(len(input_data))
input_data.head()

3842


,id,seeker_post,response_post,conversation_history
0,14_a1df3c7155d0438b9c4084b57b66c6e6_0_0,NaN,good evening I understand you're feeling isola...,[]
1,14_a1df3c7155d0438b9c4084b57b66c6e6_0_1,"Yeah, it's just... everyone was with their fam...",I hear you. It can be difficult growing apart ...,"[""Helper: good evening I understand you're fee..."
2,14_a1df3c7155d0438b9c4084b57b66c6e6_0_2,"Yeah, it's tough. You know, during the holiday...",Perhaps they're feeling similarly. Waiting for...,"[""Helper: good evening I understand you're fee..."
3,14_a1df3c7155d0438b9c4084b57b66c6e6_0_3,But why should I always have to be the bigger ...,"Being the bigger person can feel burdensome, c...","[""Helper: good evening I understand you're fee..."
4,14_a1df3c7155d0438b9c4084b57b66c6e6_0_4,"Yeah, exactly. It's like, why should I keep pu...",How would you like for them to show you they c...,"[""Helper: good evening I understand you're fee..."


In [15]:
from datasets import Dataset

study_dataset = Dataset.from_pandas(input_data)

In [8]:
from transformers import pipeline
import json

classifier_predict_config = {
    "Empathy-goodareas": {
        'model': "./roberta-Empathy-goodareas-eval_FeedbackESConv5pp_CARE10pp-sweeps-best-3lsljr3b-1741257767",
        'context_size': 1 # double check
    }
    "Reflections-goodareas": {
        'model': "./roberta-Reflections-goodareas-eval_FeedbackESConv5pp_CARE10pp-sweeps-best-d6x1jzik-1741277930",
        'context_size': 1 # double check
    },
    "Questions-goodareas": {
        'model': "./roberta-Questions-goodareas-eval_FeedbackESConv5pp_CARE10pp-sweeps-best-82jc07j0-1741329550",
        'context_size': 1 # double check
    },
    "Validation-goodareas": {
        'model': "./roberta-Validation-goodareas-eval_FeedbackESConv5pp_CARE10pp-sweeps-best-wdbkc6pj-1741680290",
        'context_size': 3,
    },
    "Suggestions-badareas": {
        'model': "./roberta-Suggestions-badareas-eval_FeedbackESConv5pp_CARE10pp-sweeps-best-qpkjg3iw-1741352101",
        'context_size': 3
    },
    "Self-disclosure-badareas": {
        'model': "./roberta-Self-disclosure-badareas-eval_FeedbackESConv5pp_CARE10pp-sweeps-best-fk58yziy-1741688349",
        'context_size': 3 # interesting, roberta suffers from 512 token max context length, so had to artificially set this, lower than what it was at. 
    },
    "Suggestions-goodareas": {
        'model': "./roberta-Suggestions-goodareas-eval_FeedbackESConv5pp_CARE10pp-sweeps-best-6xtokquj-1741444633",
        'context_size': 3
    },
}

# Set which class to use
WHICH_CLASS = "Suggestions-goodareas"

# Load the appropriate model based on configuration
classifier = pipeline("sentiment-analysis",model=classifier_predict_config[WHICH_CLASS]['model'],device=0)

def binary_prediction_seeker_response_post(conversation_history, seeker, helper, context_size=None):    
    # Use the context size from config if not provided
    if context_size is None:
        context_size = classifier_predict_config[WHICH_CLASS]['context_size']

    # Handle different input types
    if isinstance(conversation_history, str):
        history = eval(conversation_history)
    else:
        history = conversation_history
    
    history.append(f"Seeker: {seeker}")

    attempt_worked = False
    context_size_attempt = context_size
    while not attempt_worked:
        try:
            sample = f"{'\n'.join(history[-context_size_attempt:])}[SEP]Helper: {helper}"
            pred = classifier(sample)
            attempt_worked = True
            return int(pred[0]['label'] == 'selected')
        except RuntimeError as e:
            print("Sample that failed: \n", sample)
            if context_size_attempt == 1:
                # still seems to make CUDA fail, so have to manually back0out
                raise e
            else:
                context_size_attempt -= 1

Device set to use cuda:0


In [9]:
def predict_reflection(example):
    # Apply your binary prediction function to each example
    example["prediction"] = binary_prediction_seeker_response_post(
        example["conversation_history"],
        example["seeker_post"], 
        example["response_post"]
    )
    return example

# Apply the function to the entire dataset at once
predicted_dataset = study_dataset.map(predict_reflection)

Map: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3842/3842 [00:58<00:00, 65.23 examples/s]


In [11]:
input_data[f"{WHICH_CLASS}"] = predicted_dataset['prediction']

In [12]:
print(len(input_data))
input_data.head()

3842


,id,seeker_post,response_post,conversation_history,Suggestions-goodareas
0,14_a1df3c7155d0438b9c4084b57b66c6e6_0_0,NaN,good evening I understand you're feeling isola...,[],0
1,14_a1df3c7155d0438b9c4084b57b66c6e6_0_1,"Yeah, it's just... everyone was with their fam...",I hear you. It can be difficult growing apart ...,"[""Helper: good evening I understand you're fee...",0
2,14_a1df3c7155d0438b9c4084b57b66c6e6_0_2,"Yeah, it's tough. You know, during the holiday...",Perhaps they're feeling similarly. Waiting for...,"[""Helper: good evening I understand you're fee...",0
3,14_a1df3c7155d0438b9c4084b57b66c6e6_0_3,But why should I always have to be the bigger ...,"Being the bigger person can feel burdensome, c...","[""Helper: good evening I understand you're fee...",0
4,14_a1df3c7155d0438b9c4084b57b66c6e6_0_4,"Yeah, exactly. It's like, why should I keep pu...",How would you like for them to show you they c...,"[""Helper: good evening I understand you're fee...",0


In [13]:
print(f"N94_all_seekerhelper_pairs_{WHICH_CLASS}.csv")
input_data.to_csv(f"N94_all_seekerhelper_pairs_{WHICH_CLASS}.csv")

N94_all_seekerhelper_pairs_Suggestions-goodareas.csv


In [37]:
f'all_{condition}_seekerhelper_pairs_{WHICH_CLASS}.csv'

'all_control_seekerhelper_pairs_Reflections-goodareas.csv'